In [1]:
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter

#The ANFIS and other necesary functions

##Functions

###MFs

In [2]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [3]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [4]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [5]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [6]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

###Consequent Functions

In [7]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

###Other Functions

In [8]:
def sum(x):
    return torch.sum(x, dim=-1)

##Layers

In [9]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly

    Other attributes:
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f'], x_train=[]):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        if x_train == []:
            self.premises = Parameter(torch.rand(input_size, rules, len(params)), requires_grad=True)
        else:
            self.premises = Parameter(torch.rand(input_size, rules, len(params)), requires_grad=False)
            if self.rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (self.rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    for j in range(self.rules):
                        self.premises[i][j][0] = h[j]
                        self.premises[i][j][1] = stp[i]/2
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
            else:
                for i in range(self.input_size):
                    for j in range(self.rules):
                        self.premises[i][j][0] = torch.std(x_train[:, i])
                        self.premises[i][j][1] = torch.mean(x_train[:, i])
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
            self.premises.requires_grad = True

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]


In [10]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [11]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        rules : number of rules
        function : function used as a consequent function

    Other attributes:
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        self.consequents = Parameter(torch.rand(rules, input_size + 1), requires_grad=True)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [12]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

##Strucuture

In [13]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.input_size = input_size
        if x_train == []:
            self.fuzzify_layer = FuzzifyLayer(input_size, rules, mf, mf_params)
        else:
            self.fuzzify_layer = FuzzifyLayer(input_size, rules, mf, mf_params, x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

    def firing_levels(self, x):
        x = self.fuzzify_layer(x)
        fl = x.prod(dim=x.dim()-2)
        return fl

    @property
    def premises_structure(self):
        self.fuzzify_layer.premises_structure

    @property
    def premises(self):
        return self.fuzzify_layer.premises

    @property
    def consequents_structure(self):
        self.consequent_layer.consequents_structure

    @property
    def consequents(self):
        return self.consequent_layer.consequents


#Data and Datasets

In [14]:
import torch.utils.data as data

In [15]:
x_train = torch.rand(100, 2)*10

In [16]:
def function(tensor):
    results = torch.zeros(tensor.size(0))

    for i in range(tensor.size(0)):
        x0 = tensor[i, 0].item()
        x1 = tensor[i, 1].item()

        if x0 > x1:
            if int(x0) % 2 == 0:
                if int(x1) % 2 == 0:
                    results[i] = x0 * x1 / 4
                else:
                    results[i] = x0 * x1
            else:
                if int(x1) % 2 == 0:
                    results[i] = (x0 - x1) * 3
                else:
                    results[i] = (x0 - x1) * 2
        else:
            if int(x0) % 2 == 0:
                if int(x1) % 2 == 0:
                    results[i] = (x0 + x1) / 2
                else:
                    results[i] = (x0 + x1)
            else:
                if int(x1) % 2 == 0:
                    results[i] = (x1 - x0 + 2) * 2
                else:
                    results[i] = (x1 - x0 + 2)

    return results.unsqueeze(1)

In [17]:
y_train = function(x_train)
y_train.requires_grad

False

In [18]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8, shuffle = True)

In [19]:
data_iter = iter(loader)
batch1 = next(data_iter)
x_batch1, y_batch1 = batch1
x_batch1

tensor([[1.9256, 4.4712],
        [7.4889, 9.3770],
        [6.9890, 1.3485],
        [9.6538, 1.1785],
        [6.2921, 1.0522],
        [9.9347, 5.1390],
        [6.9340, 2.3172],
        [6.8273, 9.0749]])

# Gradient testing

In [20]:
model = Type3ANFIS(x_train.shape[1], x_train=x_train)

In [21]:
print(model.premises)
print(model.premises.grad)
#print(model.premises.grad_fn)
print(model.consequents)
print(model.consequents.grad)

Parameter containing:
tensor([[[0.1328, 2.4505, 2.0000],
         [5.0338, 2.4505, 2.0000],
         [9.9347, 2.4505, 2.0000]],

        [[0.1193, 2.4306, 2.0000],
         [4.9805, 2.4306, 2.0000],
         [9.8416, 2.4306, 2.0000]]], requires_grad=True)
None
Parameter containing:
tensor([[0.2201, 0.0350, 0.5028],
        [0.4964, 0.6416, 0.6449],
        [0.8575, 0.7280, 0.8721]], requires_grad=True)
None


In [29]:
trainable_parameters = [model.premises, model.consequents]

optimizer = torch.optim.SGD(trainable_parameters, lr=0.001)
prediction = model(x_batch1)
prediction.retain_grad()
prediction.requires_grad_()
loss = (y_batch1 - prediction).pow(2).sum()
print(loss)

tensor(7772.5078, grad_fn=<SumBackward0>)


In [23]:
loss.backward()

In [24]:
print(prediction)
print(prediction.grad)
#print(prediction.grad_fn)

tensor([ 4.0963, 13.9797,  4.9638,  6.1973,  4.3756, 11.2793,  5.5697, 12.7803],
       grad_fn=<SumBackward1>)
tensor([-85.4307,  72.7033, -71.5509, -51.8145, -80.9617,  29.4971, -61.8565,
         53.5132])


In [25]:
optimizer.step()
print(model.premises)
print(model.premises.grad)
print(model.consequents)
print(model.consequents.grad)

Parameter containing:
tensor([[[0.1018, 2.4016, 2.0299],
         [5.0805, 2.6087, 1.9031],
         [9.9684, 2.4098, 2.0249]],

        [[0.0751, 2.3547, 2.0461],
         [5.0180, 2.5296, 1.9399],
         [9.8980, 2.3358, 2.0576]]], requires_grad=True)
tensor([[[  31.0620,   48.8319,  -29.9152],
         [ -46.6962, -158.1973,   96.9141],
         [ -33.7139,   40.6146,  -24.8812]],

        [[  44.1521,   75.9118,  -46.1276],
         [ -37.5028,  -98.9883,   60.1499],
         [ -56.4010,   94.7535,  -57.5767]]])
Parameter containing:
tensor([[ 0.2552,  0.0801,  0.5149],
        [ 2.3577,  1.1957,  0.9603],
        [-0.1386, -0.4221,  0.7406]], requires_grad=True)
tensor([[  -35.0927,   -45.0980,   -12.0598],
        [-1861.3094,  -554.0781,  -315.4095],
        [  996.1470,  1150.0791,   131.5685]])


Both premises and consequents are updated, so its possible to start experimenting with learning algorithms.


#Experimental Learning Algorithms

# SONFIS operators

In [45]:
a = torch.tensor([[6, 5, 8],[2, 4, 6],[7, 7, 5],[9, 1, 2]])
a

tensor([[6, 5, 8],
        [2, 4, 6],
        [7, 7, 5],
        [9, 1, 2]])

In [57]:
b = torch.max(a, dim=1)
b

torch.return_types.max(
values=tensor([8, 6, 7, 9]),
indices=tensor([2, 2, 0, 0]))

In [58]:
c = b.values[b.values <= 7]
c

tensor([6, 7])

In [59]:
d = b.indices[b.values <= 7]
d

tensor([2, 0])

In [ ]:
#Ngrow and EpsilonGrow parameters its established by the user
def GrowNet(ANFISmodel, loader, Ngrow, DeltaGrow):

    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1).values

        #Boolean mask to extract the necessary info
        mask = (max_fl <= DeltaGrow**ANFISmodel.input_size)

        #The info is extracted


In [ ]:
#Ngrow and EpsilonGrow parameters its established by the user
def GrowNet(ANFISmodel, loader, Ngrow, EpsilonGrow):
    for x_batch, y_batch in loader:
        for i in range(loader.batch_size):
            firing_levels = ANFISmodel.firing_levels(x_batch[i])